# VAE로 손글씨 숫자 생성하기

**30일 딥러닝 완전 정복 · Day 26** — 강의 본문: [https://muy-bien-hwa.github.io/For-my-Claude/](https://muy-bien-hwa.github.io/For-my-Claude/)

### 이 노트북에서 할 일
- MNIST VAE를 학습해 재구성·무작위 생성·잠재 공간 보간을 시각화한다
- 잠재 차원 2짜리 VAE로 '숫자 지도'를 그린다
- β 값에 따른 변화를 비교한다

> ⚙️ **런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 선택하세요. 셀은 위에서부터 `Shift+Enter`로 실행합니다.
> 그래프 라벨은 Colab 기본 글꼴 문제를 피하려고 영어로 표기했습니다.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
device = "cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(0)
train_ds = datasets.MNIST("data", train=True, download=True, transform=transforms.ToTensor())
test_ds = datasets.MNIST("data", train=False, download=True, transform=transforms.ToTensor())
train_dl = DataLoader(train_ds, 128, shuffle=True, num_workers=2); test_dl = DataLoader(test_ds, 256)

class VAE(nn.Module):
    def __init__(self, d_z=16, d_h=512):
        super().__init__()
        self.enc = nn.Sequential(nn.Flatten(), nn.Linear(784, d_h), nn.ReLU(), nn.Linear(d_h, d_h // 2), nn.ReLU())
        self.mu, self.logvar = nn.Linear(d_h // 2, d_z), nn.Linear(d_h // 2, d_z)
        self.dec = nn.Sequential(nn.Linear(d_z, d_h // 2), nn.ReLU(), nn.Linear(d_h // 2, d_h), nn.ReLU(), nn.Linear(d_h, 784))
    def encode(self, x): h = self.enc(x); return self.mu(h), self.logvar(h)
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)        # 재매개변수화
        return self.dec(z), mu, logvar

def vae_loss(x, logits, mu, logvar, beta=1.0):
    recon = F.binary_cross_entropy_with_logits(logits, x.flatten(1), reduction="sum") / x.size(0)
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    return recon + beta * kl, recon, kl

def train_vae(d_z=16, beta=1.0, epochs=15):
    m = VAE(d_z).to(device); opt = torch.optim.Adam(m.parameters(), 1e-3)
    for ep in range(epochs):
        tot = rc = klt = 0
        for x, _ in train_dl:
            x = x.to(device); logits, mu, lv = m(x); loss, r, k = vae_loss(x, logits, mu, lv, beta)
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item(); rc += r.item(); klt += k.item()
        n = len(train_dl); print(f"[d_z={d_z} β={beta}] epoch {ep+1}: loss {tot/n:.1f}  recon {rc/n:.1f}  KL {klt/n:.1f}")
    return m
model = train_vae()

## 1. 재구성과 무작위 생성

In [ ]:
model.eval()
x, y = next(iter(test_dl)); x = x[:10].to(device)
with torch.no_grad():
    rec = torch.sigmoid(model(x)[0]).view(-1, 28, 28).cpu()
    samples = torch.sigmoid(model.dec(torch.randn(30, 16, device=device))).view(-1, 28, 28).cpu()
fig, axs = plt.subplots(2, 10, figsize=(12, 2.6))
for i in range(10):
    axs[0, i].imshow(x[i, 0].cpu(), cmap="gray"); axs[1, i].imshow(rec[i], cmap="gray"); axs[0, i].axis("off"); axs[1, i].axis("off")
axs[0, 0].set_title("input", fontsize=9, loc="left"); axs[1, 0].set_title("reconstruction", fontsize=9, loc="left"); plt.show()
fig, axs = plt.subplots(3, 10, figsize=(12, 3.8))
for a, s in zip(axs.ravel(), samples): a.imshow(s, cmap="gray"); a.axis("off")
plt.suptitle("samples from z ~ N(0, I)"); plt.show()

## 2. 잠재 공간 보간
두 숫자의 잠재 벡터 사이를 직선으로 이동하며 디코딩합니다. 중간 이미지가 그럴듯하게 이어진다면 잠재 공간이 매끄럽다는 뜻입니다.

In [ ]:
xs, ys = next(iter(test_dl))
a = xs[(ys == 1).nonzero()[0]].to(device); b = xs[(ys == 7).nonzero()[0]].to(device)
with torch.no_grad():
    za, zb = model.encode(a)[0], model.encode(b)[0]
    frames = [torch.sigmoid(model.dec((1 - t) * za + t * zb)).view(28, 28).cpu() for t in torch.linspace(0, 1, 10)]
fig, axs = plt.subplots(1, 10, figsize=(12, 1.6))
for ax, f in zip(axs, frames): ax.imshow(f, cmap="gray"); ax.axis("off")
plt.show()

## 3. 2차원 잠재 공간: 숫자 지도

In [ ]:
m2 = train_vae(d_z=2, epochs=15); m2.eval()
grid = torch.linspace(-2.5, 2.5, 15)
canvas = np.zeros((15 * 28, 15 * 28))
with torch.no_grad():
    for i, yv in enumerate(grid.flip(0)):
        for j, xv in enumerate(grid):
            img = torch.sigmoid(m2.dec(torch.tensor([[xv, yv]], device=device))).view(28, 28).cpu().numpy()
            canvas[i*28:(i+1)*28, j*28:(j+1)*28] = img
plt.figure(figsize=(8, 8)); plt.imshow(canvas, cmap="gray"); plt.axis("off"); plt.title("decoded 2D latent grid"); plt.show()
zs, ls = [], []
with torch.no_grad():
    for x, y in test_dl: zs.append(m2.encode(x.to(device))[0].cpu()); ls.append(y)
zs, ls = torch.cat(zs), torch.cat(ls)
plt.figure(figsize=(7, 6)); plt.scatter(zs[:, 0], zs[:, 1], c=ls, cmap="tab10", s=2); plt.colorbar(); plt.title("test set in latent space"); plt.show()

## 4. β 비교

In [ ]:
for beta in [0.0, 4.0]:
    mb = train_vae(beta=beta, epochs=8); mb.eval()
    with torch.no_grad(): s = torch.sigmoid(mb.dec(torch.randn(10, 16, device=device))).view(-1, 28, 28).cpu()
    fig, axs = plt.subplots(1, 10, figsize=(12, 1.6))
    for a, im in zip(axs, s): a.imshow(im, cmap="gray"); a.axis("off")
    plt.suptitle(f"beta = {beta}"); plt.show()

## 과제
- β=0(사실상 오토인코더)에서 무작위 생성 결과가 왜 이상한지 강의 내용으로 설명해 보세요.
- β=4에서 재구성 품질과 생성 다양성은 어떻게 바뀌었나요?